# (urbanair-dm) Evaluate and analyse (minimal working example)

Doppler wind-lidar (DWL) stats evaluation.

Creator: Matthias Zeeman, University of Freiburg, for the UrbanAIR project.

### Included in this Notebook

1. How to read and explore the (intermediate, final) results data (**L2**)
2. How to read and explore the (intermediate) results data (**L1**)
3. How to read and explore the source data (**L0**)

In [ ]:
import glob
import io
import itertools
import json
import math
import os
import pickle
import re
import sys
import warnings
from collections import OrderedDict, defaultdict
from pathlib import Path
import psutil

import bottleneck
import markdown
import numpy as np
import pandas as pd
import plotly.express as px
import xarray as xr
import zarr
from IPython.display import HTML
from IPython.display import Image as IPImage
from PIL import Image
from plotly.express.colors import sample_colorscale
from tqdm.notebook import tqdm

In [ ]:
# set a static renderer for notebook previews on github
import plotly.io as pio

png_renderer = pio.renderers["png"]
png_renderer.scale = 2

# for interactive plots, set default to 'notebook_connected' or 'browser'
pio.renderers.default = "png"

# Definitions

In [ ]:
sys.path.append(str(Path("../src/").resolve()))
from urbanair_dm_lib import *

# Config

In [ ]:
repository_cache_path = "../data/repositories/"  # download folder
repository_urls = [
    "https://doi.org/10.5281/zenodo.17950555",
    "https://doi.org/10.5281/zenodo.17950675",
    "https://doi.org/10.5281/zenodo.14761503",
]

repository_path = "../data/"  # source/destination folder
repository_path_L0 = [
    "../data/DWL/L0/",
]
repository_path_L1 = [
    "../data/DWL/L1/",
]
repository_path_L2 = [
    "../data/DWL/L2/",
]

# Main

## Download repositories

The following commands downloads the repositories and copies the selected files to a shared location. Skip or remove if you have manually downloaded and organised your data. This is typically only needed once, but Zenodo timeouts happen occasionally and retries are required. 

In [ ]:
if get_zenodo_repository(repository_cache_path, repository_urls):
    unpack_zenodo_repository(
        repository_cache_path,
        repository_urls,
        destination_path=repository_path,
    )

## DWL L2 (.nc)

In [ ]:
# production files
fn_list, fn_dict = input_files(repository_path_L2, "urbisphere_set*.nc")

#### Alternative

[!TIP] 
The following has been reported to be resource intensive (slow, memory footprint). As an alternative, a consolidated dataset (.zarr.zip) has been included in the Zenodo repository.

In [ ]:
if psutil.virtual_memory().available > 24*10**9:
    # consolidate production files
    ds, global_attrs_dict, _ = datastore(fn_dict)
else:
    # get a consolidated production file
    fn_list, fn_dict = input_files(
        repository_path_L2,
        "DWL_L2*.zarr.zip",
    )

    global_attrs_dict = (
        global_attrs_dict
        if "global_attrs_dict" in globals()
        else {"": {"license": "See source files!"}}
    )
    ds = xr_reindex(xr.open_dataset(fn_list[0], engine="zarr"))

### Examples:

Plot a variable with time against model level for all locations as plot panels sorted by latitude.

In [ ]:
# dataset restructure needed for plots
subset = dict()
isubset = dict(
    bounds=0,  # ... default boundary (0=left)
    time_delta=0,  # ... default aggregation period (0=10min, 1=60min, ...)
)

In [ ]:
# reduce to a 4-dimensional dataset
plot_ds = (
    ds.sel(subset)
    .isel(isubset)
    .unstack(["cell"])
    .reset_index(["station", "system"])
    .stack(location=["station", "system"])
    .dropna("location", how="all")
    .dropna("cell_id", how="all")
    .dropna("time", how="all")
    .sortby("station_lat", ascending=False)
    .transpose(..., "time")
)

In [ ]:
plot_ds

#### Wind speed

In [ ]:
var = "ws"  # wind speed

subset = dict(
    channel_id=3,  # ...  model configuration (3=MESONH model)
    cell_mode=1,  # ... sampling mode (1=half-level)
    channel_mode=1,  # ... channel from stats computations (0=stare,1=vad)
)
isubset = dict(cell_id=slice(13, None))  # ... exclude lowest levels
px_opts = dict(
    range_color=[0, 10],
    color_continuous_scale="agsunset",
)

fig = ds_plot(plot_ds, data_var=var, px_opts=px_opts, isel=isubset, sel=subset)
fig.show(height=800)

#### Vertical air velocity

In [ ]:
var = "w_mean"  # vertical air velocity

subset = dict(
    channel_id=3,  # ...  model configuration (3=MESONH model)
    cell_mode=1,  # ... sampling mode (1=half-level)
    channel_mode=0,  # ... channel from stats computations (0=stare,1=vad)
)
isubset = dict(
    cell_id=slice(13, None),  # ... exclude lowest levels
)
px_opts = dict(
    range_color=[-1, 1],
    color_continuous_scale="RdBu_r",
)

fig = ds_plot(plot_ds, data_var=var, px_opts=px_opts, isel=isubset, sel=subset)
fig.show(height=800)

#### Quality criteria

In the above example, a threshold for the relative number of samples is used to mask ambiguous data. In reality, the absolute number of samples differs per location and system due to configuration variations. These and other quality criteria must be explored further. 

In [ ]:
var = "w_count"  # number of samples

subset = dict(
    channel_id=3,  # ...  model configuration (3=MESONH model)
    cell_mode=1,  # ... sampling mode (1=half level)
    channel_mode=0,  # ... channel from stats computations (0=stare,1=vad)
)
isubset = dict(
    cell_id=slice(13, None),  # ... exclude lowest levels
)
px_opts = dict(
    range_color=[0, 600],
    color_continuous_scale="haline_r",
    facet_col_wrap=4,
)

fig = ds_plot(plot_ds, data_var=var, px_opts=px_opts, isel=isubset, sel=subset)
fig.show(height=300)

#### Quiz: Which ...?
- system took breaks during the afternoon (:zzz:)
- system was located on a high-rise building (:office:)
- system was focussed on a far-away distance (:cloud:) 
- system was focussed on a far-away distance and a short sample interval (:ghost:)

#### Swap coordinates (y-axis)

The `cell_id` coordinate is defined differently for each model configuration and may decrease with height. For this and other purposes, it may be more useful to plot altitude above the surface (i.e. height) on the y-axis.

In [ ]:
var = "wdir"  # wind direction
ycoord = ("cell_id", "cell_z_bounds")

subset = dict(
    channel_id=2,  # ...  model configuration (2=AROME model)
    cell_mode=2,  # ... sampling mode (2=entire level)
    channel_mode=1,  # ... channel from stats computations (0=stare,1=vad)
)
isubset = dict(
    cell_id=slice(13, None),  # ... exclude lowest levels
    location=[1],
)
px_opts = dict(
    range_color=[0, 360],
    color_continuous_scale="icefire",
)

fig = ds_plot(
    plot_ds,
    data_var=var,
    px_opts=px_opts,
    isel=isubset,
    sel=subset,
    ycoord=ycoord,
)
fig.show(height=300)

#### Generic model

A generic model is included, in which all the computed variables are aligned to the minimum cell size in relation to the observation resolution. This allows for easy comparison between vertical and horizontal vectors, for example. 

In [ ]:
var = "wdir"
ycoord = ("cell_id", "cell_z_bounds")

subset = dict(
    channel_id=1,  # ...  model configuration (1=station model)
    cell_mode=2,  # ... sampling mode (2=full level)
    channel_mode=1,  # ... channel from stats computations (0=stare,1=vad)
    time=slice("2023-08-21 07:00:00", "2023-08-21 17:00:00"),
)
isubset = dict(
    cell_id=slice(13, None),  # ... exclude lowest levels
    location=[0, 1, 2, 3],
)
px_opts = dict(
    range_color=[0, 360],
    color_continuous_scale="icefire",
    facet_col_wrap=4,
)

fig = ds_plot(
    plot_ds,
    data_var=var,
    px_opts=px_opts,
    isel=isubset,
    sel=subset,
    ycoord=ycoord,
)
fig.show(height=300)

#### Cell method 

In [ ]:
var = "w_std"
ycoord = ("cell_id", "cell_z_bounds")

subset = dict(
    channel_mode=0,  # ... stats computation mode
    time=slice("2023-08-21 07:00:00", "2023-08-21 17:00:00"),
)
isubset = dict()
px_opts = dict(
    facet_col_wrap=3,
    facet_col="location",
    range_color=[0, 2],
    color_continuous_scale="purples",
)

In [ ]:
# init
fh = {}
psubset = [1, 2, 3]
ptext = [
    f"model = {plot_ds['channel_id'].attrs['flag_meaning'].split(' ')[n]}"
    for n in psubset
]

# collect
n = 0
for idx, grp in plot_ds.groupby(["channel_id"]):
    if all([n in psubset for n in grp["channel_id"].data.tolist()]):
        plot_grp = (
            grp.isel(location=1)
            .unstack("channel")
            .stack(
                location=(
                    "channel_id",
                    "cell_mode",
                )
            )
        )
        fig = ds_plot(
            plot_grp,
            data_var=var,
            px_opts=px_opts,
            isel=isubset,
            sel=subset,
            ycoord=ycoord,
        )
        fh[n] = fig
        n = n + 1

# merge, patch
for n in fh.keys():
    for m in ["x", "y", "z"]:
        fig["data"][n][m] = fh[n]["data"][2][m]
    fig = fig.update_annotations(
        selector={"text": fig.to_dict()["layout"]["annotations"][n]["text"]},
        text=ptext[n],
    )

fig.show(height=300)

### Metadata

Information about the data is embeded. However, note that the `xarray` merge commands above remove non-identical attributes, including authors, licence and terms of use. Query the individual file sets for detail.

In [ ]:
with xr.set_options(display_expand_attrs=False, display_expand_data_vars=False):
    # display(HTML(ds_dict[next(iter(ga_dict))]._repr_html_()))
    display(
        HTML(
            json.dumps(
                {
                    f"<b>{k}</b>": markdown.markdown(v)
                    .replace("\n", " ")
                    .replace("<p>", "")
                    .replace("</p>", "")
                    .encode("ascii", "xmlcharrefreplace")
                    .decode("ascii")
                    for k, v in global_attrs_dict[next(iter(global_attrs_dict))].items()
                },
                indent=4,
                separators=(",<br>", ": "),
            )
        )
    )

## DWL L1 (.nc)

If higher resolution is needed:

The L1 dataset contains data in original sample resolution and uses the CF-1.10 conventions to align the vocabulary, as is done in *urbisphere* (urbisphere-dm). Currently, the L1 dataset includes vertical air velocity without auxiliary variables, with curation similar to the L0 DWL examples below.

### Examples
#### Vertical air velocity (high-res)

In [ ]:
# production files
_, fn_dict = input_files(repository_path_L1, "urbisphere_set*.nc")

In [ ]:
# read data
ds_list = []

# example, concatenate some files:
for fn_path, fn_list in fn_dict.items():
    for fn in fn_list:
        if "PACHEM" in fn_path:
            # netcdf lock workaround
            with xr.open_dataset(fn, decode_timedelta=True) as dx:
                ds0 = dx.load()
                dx.close()
            ds_list.append(ds0)

ds1 = xr.concat(ds_list, dim="time")
ds1 = xr_reindex(ds1)

In [ ]:
subset = dict(
    time=slice("2023-08-21 08:00:00", "2023-08-21 16:00:00"),
)
isubset = dict(
    cell=slice(0, 166),
)
alt_subset = dict(
    cell_z_bounds=slice(200.0, 2200.0),
)

### Example, aligned with `system`

In [ ]:
# example: extract all upward air velocity data, up to 166 range gates,
# and, normalize to a fixed time interval for comparison between locations
da = (
    ds1["w"]
    .resample(time="5s")
    .mean(skipna=False)
    .isel(isubset)
    .sel(subset)
    .squeeze()
    .dropna(dim="time", how="all")
    .transpose(..., "time")
)

fig = plot_da(da)
fig.show()

In [ ]:
# example: same, but resample to hourly intervals without heuristics for sample density
da = (
    ds1["w"]
    .resample(time="60min")
    .mean(skipna=True)
    .isel(isubset)
    .sel(subset)
    .squeeze()
    .dropna(dim="time", how="all")
    .transpose(..., "time")
)

fig = plot_da(da)
fig.show()

### Example, aligned with `station`

In [ ]:
# translate DWL L1 coordinates to `model` coordinates, using information from the DWL L2 product
df1_cell_lut, df1_coords_lut, _ = translate_channels(ds, a="system", b="station")
ds1_trans = swap_channels(ds1, df1_cell_lut)
ds1_trans = swap_coords(ds1_trans, df1_coords_lut, coord=("cell", "cell_z_bounds"))

# Recomputation of statistics
# - Not simultaneously on the time and spatial dimensions, as is done in DWL L2.
# - Without quality control, these results cannot (!) be used for publication.
da = (
    ds1_trans["w"]
    .sel(subset)
    .resample(time="5s")
    .mean(skipna=False)
    .groupby("cell_z_bounds")
    .mean()
    .dropna(dim="time", how="all")
    .sel(alt_subset)
)
da = da.rename({"cell_z_bounds": "cell_z"})  # ... as we reduced the `bounds` dim.
da = da.squeeze().sortby("cell_z").transpose(..., "time")

# Figure
# ... with custom plotly, now height in [m]
fig = plot_da(da)
fig.show()

### Example, aligned with `arome`

In [ ]:
# translate DWL L1 coordinates to `model` coordinates, using information from the DWL L2 product
df1_cell_lut, df1_coords_lut, _ = translate_channels(ds, a="system", b="arome")
ds1_trans = swap_channels(ds1, df1_cell_lut)
ds1_trans = swap_coords(ds1_trans, df1_coords_lut, coord=("cell", "cell_z_bounds"))

# Recomputation of statistics
# - Not simultaneously on the time and spatial dimensions, as is done in DWL L2.
# - Without quality control, these results cannot (!) be used for publication.
da = (
    ds1_trans["w"]
    .sel(subset)
    .resample(time="1min")
    .mean(skipna=False)
    .groupby("cell_z_bounds")
    .mean()
    .dropna(dim="time", how="all")
    .sel(alt_subset)
)
da = da.rename({"cell_z_bounds": "cell_z"})  # ... as we reduced the `bounds` dim.
da = da.squeeze().sortby("cell_z").transpose(..., "time")

# Figure
# ... with custom plotly, now height in [m]
fig = plot_da(da)
fig.show()

### Example, aligned with `mesonh`

In [ ]:
# translate DWL L1 coordinates to Model coordinates, using information from the DWL L2 product
df1_cell_lut, df1_coords_lut, _ = translate_channels(ds, a="system", b="mesonh")
ds1_trans = swap_channels(ds1, df1_cell_lut)
ds1_trans = swap_coords(ds1_trans, df1_coords_lut, coord=("cell", "cell_z_bounds"))

# Recomputation of statistics
# - Not simultaneously on the time and spatial dimensions, as is done in DWL L2.
# - Without quality control, these results cannot (!) be used for publication.
da = (
    ds1_trans["w"]
    .sel(subset)
    .resample(time="1min")
    .mean(skipna=False)
    .groupby("cell_z_bounds")
    .mean()
    .dropna(dim="time", how="all")
    .sel(alt_subset)
)
da = da.rename({"cell_z_bounds": "cell_z"})  # ... we reduced the `bounds` dim...
da = da.squeeze().sortby("cell_z").transpose(..., "time")

# Figure
# ... with custom plotly, now height in [m]
fig = plot_da(da)
fig.show()

In [ ]:
# Figure
# ... similar, with standard mathplotlib library for visualisation, 
da.plot(
    cmap="RdBu_r",    
    vmin=-2,
    vmax=2,
    figsize=(18, 5),
)

In [ ]:
def l1_consolidated(destination=["station",'mesonh','arome']):
    repository_path_L1 = [
        "../data/DWL/L1/",
    ]
    repository_path_L2 = [
        "../data/DWL/L2/",
    ]
    
    # L2 get a consolidated production file
    fn_list, fn_dict = input_files(repository_path_L2, "DWL_L2*.zarr.zip")
    ds = xr_reindex(xr.open_dataset(fn_list[0], engine="zarr"))
    
    # L1 production files
    _, fn_dict = input_files(repository_path_L1, "urbisphere_set*.nc")
    dt_dict = {}
    # example, concatenate some files:
    for b in tqdm(destination):    
        for fn_path, fn_list in tqdm(fn_dict.items()):
            # read
            ds_list = []
            for fn in fn_list:
                # netcdf lock workaround
                with xr.open_dataset(fn, decode_timedelta=True) as dx:
                    ds0 = dx.load()
                    dx.close()
                ds_list.append(ds0)
        
            ds1 = xr.concat(ds_list, dim="time")
            ds1 = xr_reindex(ds1)
        
            # translate
            df1_cell_lut, df1_coords_lut, _ = translate_channels(ds, a="system", b=b)
            ds1_trans = swap_channels(ds1, df1_cell_lut)
            ds1_trans = swap_coords(ds1_trans, df1_coords_lut, coord=("cell", "cell_z_bounds"))
    
            # cleanup
            ds1_trans = ds1_trans.groupby("cell_z_bounds").mean()  
        
            # store
            ix = "".join(ds1["station_id"].values.tolist())
            gx = f"{b}/{ix[2:]}"
            dt_dict[gx] = ds1_trans

    
    f0_localstore(
        "../tmp/DWL_L1_data_consolidated.zarr.zip",
        mode="w",
        data={
            k: ds.reset_index([n for n in ds.indexes if ds.indexes.is_multi(n)])
            for k, ds in dt_dict.items()
        },
    )

l1_consolidated()

### Metadata

In [ ]:
# infos about data variables
HTML(json.dumps({f"<b>{ds1[dv].name}</b>": ds1[dv].attrs for dv in ds1.data_vars}))

In [ ]:
# display metadata context, if applicable
HTML(
    json.dumps(
        {
            f"<b>{k}</b>": markdown.markdown(v)
            .replace("\n", " ")
            .replace("<p>", "")
            .replace("</p>", "")
            .encode("ascii", "xmlcharrefreplace")
            .decode("ascii")
            for k, v in ds1.attrs.items()
        },
        indent=4,
        separators=(",<br>", ": "),
    )
)

## DWL L0 (.nc)

L0 data were converted from RAW data and contain information from the source files and metadata databases. The variable names are aligned to the vocabulary from the origin (the file header, the operating manual, the data provider). L0 data are identical to the source, where possible. No post-calibration was applied. 

L0 DWL data archives contain HaloPhotonics (now Lumibird) model StreamLine XP files (.hpl) for a location and time period. Added are metadata required for reuse, including geographic coordinates, license terms, history and production information. The L0 data contain information about the radial velocity in the native resolution of the acquisition, e.g., approx 0.3 to 1Hz resolution for the radial ("Doppler") velocity shown below.

In [ ]:
# production files
_, fn_dict = input_files(repository_path_L0, "urbisphere_set*.nc")

In [ ]:
# read data
ds_list = []

# example, concatenate some files:
for fn_path, fn_list in fn_dict.items():
    for fn in fn_list:
        if "PACHEM" in fn_path and ",20230821T" in fn:
            # netcdf lock workaround
            with xr.open_dataset(fn, decode_timedelta=True) as dx:
                ds0 = dx.load()
                dx.close()
            ds_list.append(ds0)

ds0 = xr.concat(ds_list, dim="time")
ds0 = ds0.set_xindex('cell_id')

In [ ]:
# infos about data variables
HTML(json.dumps({f"<b>{ds0[dv].name}</b>": ds0[dv].attrs for dv in ds0.data_vars}))

### Examples
#### All-mode scans:

The L0 contains both the Stare (zenith angle) and other scan mode data. In the example below the Velocity Azimuth Display (VAD) scans can be seen as banding at 10-min intervals. 

In [ ]:
subset = dict(
    time=slice("2023-08-21 08:00:00", "2023-08-21 16:00:00"),
)
isubset = dict(
    cell=slice(0, 166),  # about 2700-3000m above sensor platform
)

In [ ]:
# extract radial velocity data
da = ds0["Doppler"].squeeze().transpose("cell", "time").sel(subset).isel(isubset)

fig = plot_da(da)
fig.show()

#### Stare mode scan:

In the example below, only data from the 'Stare' mode is extracted. These are the data recorded at a zenith angle. 

A cell unit has a length of 18.0 m (range gate length) in this example.

In [ ]:
global_attrs_dict

In [ ]:
# subset vertical wind data
scantype_key = ds0["attributes_id"] == ["Scan type"]
scantype_value = ds0["attributes_values"].isel(attributes=scantype_key)

ds_stare = (
    ds0.drop_dims("attributes")
    .where(scantype_value.isin(['"Stare"']))
    .squeeze()
    .transpose("cell", "time")
    .sel(subset)
    .isel(isubset)
    .dropna(dim="time", how="all")
)

# ... and the radial velocity data array
da_w = ds_stare["Doppler"]

In [ ]:
# Show only vertical (zenith angle) beam data
fig = plot_da(da_w)
fig.show()

#### Noise filter:

Noise is an inherent feature of LiDAR, as can be seen clearly when the return signal drops with range. Unplausible values must be removed before comparison with model results. 

Here, we apply a generic despiking filter to the cell (beam interval, or range gate) and the time domain. Values are masked with steps of ±4 m/s between consecutive observations. 

In literature, the signal-to-noise ratio is used as a quality threshold. Below, we demonstrate the impact of combining the filters.

In [ ]:
_, da_qc_mask_1 = xr_filter_spikes(
    da_w, kernel={"cell": 3, "time": 3}, filter_bnds=slice(-4, 4)
)

da_qc_mask_2 = np.logical_or(
    ds_stare["Intensity"] < (10 ** (-21 / 10) + 1),  # -21 db SNR lower thresholds
    ds_stare["Intensity"] > (10 ** (-7 / 10) + 1),  # -7 db SNR upper thresholds
)

In [ ]:
# spike filter applied
da_qc = da_w.where((~da_qc_mask_1.values))

fig = plot_da(da_qc)
fig.show()

In [ ]:
# be more stringent; spike and snr filter applied
da_qc = da_w.where((~da_qc_mask_1.values) & (~da_qc_mask_2.values))

fig = plot_da(da_qc)
fig.show()

The differences in the results obtained by applying different filters are an interesting topic for academic discussion. :^)

### Metadata

In [ ]:
# jupyter notebook context
HTML(hello_world())